In [1]:
# ============================================
# 1. Imports & Google Drive
# ============================================

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

Mounted at /content/drive


In [2]:
# ============================================
# 2. Load Raw FAOSTAT Dataset
# ============================================

file_path = "/content/drive/MyDrive/GDG Al-Zaher || Data Science || 2026/Final Project/Raw Data/FAOSTAT_data_en_9-24-2026.csv"

df = pd.read_csv(
    file_path,
    on_bad_lines="skip",
    engine="python"
)

print("Raw dataset shape:", df.shape)

display(df.head())

Raw dataset shape: (22568, 15)


,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
0,QCL,Crops and livestock products,818,Egypt,5312,Area harvested,01654,"Anise, badian, coriander, cumin, caraway, fenn...",1961,1961,ha,1200.0000,E,Estimated value,NaN
1,QCL,Crops and livestock products,818,Egypt,5412,Yield,01654,"Anise, badian, coriander, cumin, caraway, fenn...",1961,1961,kg/ha,833.3000,E,Estimated value,NaN
2,QCL,Crops and livestock products,818,Egypt,5510,Production,01654,"Anise, badian, coriander, cumin, caraway, fenn...",1961,1961,t,1000.0000,E,Estimated value,NaN
3,QCL,Crops and livestock products,818,Egypt,5312,Area harvested,01654,"Anise, badian, coriander, cumin, caraway, fenn...",1962,1962,ha,1200.0000,E,Estimated value,NaN
4,QCL,Crops and livestock products,818,Egypt,5412,Yield,01654,"Anise, badian, coriander, cumin, caraway, fenn...",1962,1962,kg/ha,833.3000,E,Estimated value,NaN


In [3]:
# ============================================
# 3. Filter Egypt
# ============================================

df = df[df["Area"] == "Egypt"].copy()

print("Egypt dataset shape:", df.shape)

print("\nMissing values:")
print(df.isna().sum())

Egypt dataset shape: (22568, 15)

Missing values:
Domain Code             0
Domain                  0
Area Code (M49)         0
Area                    0
Element Code            0
Element                 0
Item Code (CPC)         0
Item                    0
Year Code               0
Year                    0
Unit                    0
Value                 288
Flag                    0
Flag Description        0
Note                22114
dtype: int64


In [4]:
# ============================================
# 4. Keep Relevant Agricultural Elements
# ============================================

relevant_elements = [
    "Area harvested",
    "Production",
    "Yield"
]

crop = df[
    df["Element"].isin(relevant_elements)
].copy()

print("Filtered dataset shape:", crop.shape)

print("\nElements:")
print(crop["Element"].value_counts())

Filtered dataset shape: (18118, 15)

Elements:
Element
Production        8680
Area harvested    4812
Yield             4626
Name: count, dtype: int64


In [5]:
# ============================================
# 5. Pivot Dataset
# ============================================

pivot = crop.pivot_table(
    index=["Item", "Year"],
    columns="Element",
    values="Value",
    aggfunc="first"
).reset_index()

pivot.columns.name = None

print("Pivot shape:", pivot.shape)

print("\nColumns:")
print(pivot.columns.tolist())

display(pivot.head())

Pivot shape: (8498, 5)

Columns:
['Item', 'Year', 'Area harvested', 'Production', 'Yield']


,Item,Year,Area harvested,Production,Yield
0,"Anise, badian, coriander, cumin, caraway, fenn...",1961,1200.0000,1000.0000,833.3000
1,"Anise, badian, coriander, cumin, caraway, fenn...",1962,1200.0000,1000.0000,833.3000
2,"Anise, badian, coriander, cumin, caraway, fenn...",1963,1200.0000,1000.0000,833.3000
3,"Anise, badian, coriander, cumin, caraway, fenn...",1964,1200.0000,1000.0000,833.3000
4,"Anise, badian, coriander, cumin, caraway, fenn...",1965,1200.0000,1000.0000,833.3000


In [6]:
# ============================================
# 6. Duplicate Check
# ============================================

duplicates = pivot.duplicated(
    subset=["Item", "Year"]
).sum()

print("Duplicate Item-Year records:", duplicates)

Duplicate Item-Year records: 0


In [7]:
# ============================================
# 7. Invalid Value Check
# ============================================

numeric_columns = [
    "Area harvested",
    "Production",
    "Yield"
]

print("Negative values:")

for col in numeric_columns:
    print(
        f"{col}:",
        (pivot[col] < 0).sum()
    )

Negative values:
Area harvested: 0
Production: 0
Yield: 0


In [8]:
# ============================================
# 8. Missing Value Check
# ============================================

print(pivot.isna().sum())

Item                 0
Year                 0
Area harvested    3856
Production           0
Yield             3872
dtype: int64


In [9]:
# ============================================
# 9. Sort by Crop and Year
# ============================================

df_ml = pivot.copy()

df_ml = (
    df_ml
    .sort_values(["Item", "Year"])
    .reset_index(drop=True)
)

display(df_ml.head())

,Item,Year,Area harvested,Production,Yield
0,"Anise, badian, coriander, cumin, caraway, fenn...",1961,1200.0000,1000.0000,833.3000
1,"Anise, badian, coriander, cumin, caraway, fenn...",1962,1200.0000,1000.0000,833.3000
2,"Anise, badian, coriander, cumin, caraway, fenn...",1963,1200.0000,1000.0000,833.3000
3,"Anise, badian, coriander, cumin, caraway, fenn...",1964,1200.0000,1000.0000,833.3000
4,"Anise, badian, coriander, cumin, caraway, fenn...",1965,1200.0000,1000.0000,833.3000


In [10]:
# ============================================
# 10. Historical Lag Features
# ============================================

# Yield
df_ml["Yield_Lag1"] = (
    df_ml.groupby("Item")["Yield"].shift(1)
)

df_ml["Yield_Lag2"] = (
    df_ml.groupby("Item")["Yield"].shift(2)
)

# Production
df_ml["Production_Lag1"] = (
    df_ml.groupby("Item")["Production"].shift(1)
)

df_ml["Production_Lag2"] = (
    df_ml.groupby("Item")["Production"].shift(2)
)

# Area
df_ml["Area_Lag1"] = (
    df_ml.groupby("Item")["Area harvested"].shift(1)
)

df_ml["Area_Lag2"] = (
    df_ml.groupby("Item")["Area harvested"].shift(2)
)

In [11]:
# ============================================
# 11. Rolling Yield Feature
# ============================================

df_ml["Yield_RollingMean_3Y"] = (
    df_ml
    .groupby("Item")["Yield"]
    .transform(
        lambda x: x.shift(1).rolling(3).mean()
    )
)

In [12]:
# ============================================
# 12. Historical Growth Features
# ============================================

df_ml["Area_Growth_Lag1"] = (
    df_ml["Area_Lag1"] /
    df_ml["Area_Lag2"] - 1
)

df_ml["Production_Growth_Lag1"] = (
    df_ml["Production_Lag1"] /
    df_ml["Production_Lag2"] - 1
)

In [13]:
# ============================================
# 13. Define Features and Target
# ============================================

numerical_features = [
    "Yield_Lag1",
    "Yield_Lag2",
    "Production_Lag1",
    "Area_Lag1",
    "Yield_RollingMean_3Y",
    "Area_Growth_Lag1",
    "Production_Growth_Lag1"
]

categorical_features = [
    "Item"
]

target = "Yield"

final_columns = (
    categorical_features +
    ["Year"] +
    numerical_features +
    [target]
)

ml_data = df_ml[final_columns].copy()

print("Before removing incomplete records:")
print("Shape:", ml_data.shape)

display(ml_data.head())

Before removing incomplete records:
Shape: (8498, 10)


,Item,Year,Yield_Lag1,Yield_Lag2,Production_Lag1,Area_Lag1,Yield_RollingMean_3Y,Area_Growth_Lag1,Production_Growth_Lag1,Yield
0,"Anise, badian, coriander, cumin, caraway, fenn...",1961,NaN,NaN,NaN,NaN,NaN,NaN,NaN,833.3000
1,"Anise, badian, coriander, cumin, caraway, fenn...",1962,833.3000,NaN,1000.0000,1200.0000,NaN,NaN,NaN,833.3000
2,"Anise, badian, coriander, cumin, caraway, fenn...",1963,833.3000,833.3000,1000.0000,1200.0000,NaN,0.0000,0.0000,833.3000
3,"Anise, badian, coriander, cumin, caraway, fenn...",1964,833.3000,833.3000,1000.0000,1200.0000,833.3000,0.0000,0.0000,833.3000
4,"Anise, badian, coriander, cumin, caraway, fenn...",1965,833.3000,833.3000,1000.0000,1200.0000,833.3000,0.0000,0.0000,833.3000


In [14]:
# ============================================
# 14. Remove Incomplete ML Records
# ============================================

before = len(ml_data)

ml_data = ml_data.dropna(
    subset=numerical_features + [target]
).copy()

after = len(ml_data)

print("Records before:", before)
print("Records after:", after)
print("Removed:", before - after)

print("\nFinal missing values:")
print(ml_data.isna().sum())

Records before: 8498
Records after: 4329
Removed: 4169

Final missing values:
Item                      0
Year                      0
Yield_Lag1                0
Yield_Lag2                0
Production_Lag1           0
Area_Lag1                 0
Yield_RollingMean_3Y      0
Area_Growth_Lag1          0
Production_Growth_Lag1    0
Yield                     0
dtype: int64


In [15]:
# ============================================
# 15. Final Duplicate Check
# ============================================

print(
    "Duplicate Item-Year records:",
    ml_data.duplicated(
        subset=["Item", "Year"]
    ).sum()
)

Duplicate Item-Year records: 0


In [16]:
# ============================================
# 16. Final Dataset Summary
# ============================================

print("Final ML Dataset Shape:", ml_data.shape)

print("\nNumber of crops:")
print(ml_data["Item"].nunique())

print("\nYear range:")
print(
    ml_data["Year"].min(),
    "to",
    ml_data["Year"].max()
)

print("\nTarget statistics:")
display(
    ml_data["Yield"].describe()
)

print("\nRecords per crop:")
display(
    ml_data["Item"].value_counts().describe()
)

Final ML Dataset Shape: (4329, 10)

Number of crops:
77

Year range:
1964 to 2024

Target statistics:


,Yield
count,4329.0000
mean,13780.6681
std,15056.9818
min,666.7000
25%,2871.9000
50%,9787.7000
75%,20529.6000
max,121348.8000



Records per crop:


,count
count,77.0000
mean,56.2208
std,11.2912
min,1.0000
25%,61.0000
50%,61.0000
75%,61.0000
max,61.0000


In [17]:
# ============================================
# 17. Outlier Detection
# ============================================

Q1 = ml_data["Yield"].quantile(0.25)
Q3 = ml_data["Yield"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = ml_data[
    (ml_data["Yield"] < lower_bound) |
    (ml_data["Yield"] > upper_bound)
].copy()

print("Yield lower bound:", lower_bound)
print("Yield upper bound:", upper_bound)

print("\nNumber of potential outliers:", len(outliers))

print(
    f"Percentage of potential outliers: "
    f"{len(outliers) / len(ml_data) * 100:.2f}%"
)

Yield lower bound: -23614.649999999994
Yield upper bound: 47016.149999999994

Number of potential outliers: 93
Percentage of potential outliers: 2.15%


In [18]:
# ============================================
# 18. Time-Based Train/Test Split
# ============================================

split_year = int(
    ml_data["Year"].quantile(0.80)
)

train_data = ml_data[
    ml_data["Year"] <= split_year
].copy()

test_data = ml_data[
    ml_data["Year"] > split_year
].copy()

print("Split year:", split_year)

print("\nTrain shape:", train_data.shape)
print("Test shape:", test_data.shape)

print(
    "\nTrain years:",
    train_data["Year"].min(),
    "to",
    train_data["Year"].max()
)

print(
    "Test years:",
    test_data["Year"].min(),
    "to",
    test_data["Year"].max()
)

Split year: 2013

Train shape: (3502, 10)
Test shape: (827, 10)

Train years: 1964 to 2013
Test years: 2014 to 2024


In [19]:
# ============================================
# 19. Prepare X and y
# ============================================

X_train = train_data[
    categorical_features +
    numerical_features
].copy()

y_train = train_data[target].copy()

X_test = test_data[
    categorical_features +
    numerical_features
].copy()

y_test = test_data[target].copy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (3502, 8)
y_train: (3502,)
X_test: (827, 8)
y_test: (827,)


In [20]:
# ============================================
# 20. Preprocessing Pipeline
# ============================================

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numerical_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [21]:
# ============================================
# 21. Fit Preprocessor on Training Data ONLY
# ============================================

X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

print(
    "Processed X_train shape:",
    X_train_processed.shape
)

print(
    "Processed X_test shape:",
    X_test_processed.shape
)

Processed X_train shape: (3502, 83)
Processed X_test shape: (827, 83)


In [22]:
# ============================================
# 22. Final Validation
# ============================================

print("========== FINAL PREPROCESSING CHECK ==========")

print("\nOriginal ML dataset:")
print(ml_data.shape)

print("\nTrain:")
print(X_train.shape, y_train.shape)

print("\nTest:")
print(X_test.shape, y_test.shape)

print("\nProcessed Train:")
print(X_train_processed.shape)

print("\nProcessed Test:")
print(X_test_processed.shape)

print("\nTrain/Test year ranges:")

print(
    "Train:",
    train_data["Year"].min(),
    "->",
    train_data["Year"].max()
)

print(
    "Test:",
    test_data["Year"].min(),
    "->",
    test_data["Year"].max()
)

========== FINAL PREPROCESSING CHECK ==========

Original ML dataset:
(4329, 10)

Train:
(3502, 8) (3502,)

Test:
(827, 8) (827,)

Processed Train:
(3502, 83)

Processed Test:
(827, 83)

Train/Test year ranges:
Train: 1964 -> 2013
Test: 2014 -> 2024


In [23]:
# ============================================
# 23. Save Preprocessed Datasets
# ============================================

output_path = "/content/drive/MyDrive/GDG Al-Zaher || Data Science || 2026/Final Project/Preprocessed Data"

import os
os.makedirs(output_path, exist_ok=True)

ml_data.to_csv(
    f"{output_path}/ml_dataset.csv",
    index=False
)

train_data.to_csv(
    f"{output_path}/train_data.csv",
    index=False
)

test_data.to_csv(
    f"{output_path}/test_data.csv",
    index=False
)

print("Preprocessed datasets saved successfully.")

Preprocessed datasets saved successfully.
